# Séance 7 — Préparer les données pour le machine learning

## Objectifs

On suit toute la chaîne de préparation des données :

**données brutes → exploration → valeurs manquantes → imputation → variables catégorielles → encodage → mise à l'échelle → `ColumnTransformer` → `Pipeline` → évaluation**

On utilise un jeu de données synthétique dont nous connaissons la mécanique de génération. La probabilité d'achat dépend de plusieurs variables avec des **effets non linéaires**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
n = 2000

age = np.random.uniform(18, 75, n)
revenu = np.random.lognormal(mean=np.log(2800), sigma=0.35, size=n)
revenu = np.clip(revenu, 1200, 9000)
anciennete = np.clip(np.random.exponential(scale=3, size=n), 0, 15)
visites = np.random.poisson(lam=5, size=n)
distance = np.clip(np.random.gamma(shape=2.5, scale=4, size=n), 0.2, 40)

ville = np.random.choice(["Tours", "Paris", "Lyon", "Nantes"], n,
                          p=[0.30, 0.30, 0.20, 0.20])
statut = np.random.choice(["Nouveau", "Régulier", "Fidèle"], n,
                          p=[0.35, 0.45, 0.20])

df = pd.DataFrame({
    "age": age, "revenu": revenu, "anciennete": anciennete,
    "visites": visites, "distance": distance,
    "ville": ville, "statut": statut
})

df.head()

## 1. Construire la probabilité d'achat

La relation réelle n'est pas simplement linéaire.

- le revenu augmente l'intérêt puis sature ;
- les visites ont un effet de seuil ;
- l'âge a un effet en cloche ;
- la distance diminue l'intérêt ;
- ville et statut ont des effets différents.

La fonction logistique transforme ensuite un score en probabilité. Elle ne signifie donc pas que les effets des variables sont linéaires.

Le score utilisé est :

$$ \boxed{ s = -2.4 +1.2\,g_R(R) +g_V(V) +g_A(A) -0.09D +g_{\text{ville}} +g_{\text{statut}} } $$

avec :

Effet du revenu
$$ g_R(R) = \min\left(1.5,\max\left(0,\frac{R-2000}{2500}\right)\right). $$

Donc l'effet du revenu augmente jusqu'à un certain niveau puis sature.

Effet du nombre de visites
$$ g_V(V) = \frac{1.4}{1+\exp(-(V-5))}. $$

C'est un effet de seuil autour de 5 visites.

Effet de l'âge
$$ g_A(A) = 1.5\exp\left( -\frac{(A-40)^2}{2\times12^2} \right). $$

C'est un effet en cloche, maximal autour de 40 ans.

Effet de la distance
$$ -0.09D. $$

Plus la distance augmente, plus le score diminue.

Effets catégoriels

Pour la ville :

$$ g_{\text{ville}}= \begin{cases} 0.5 & \text{Tours}\\ 0.2 & \text{Paris}\\ -0.1 & \text{Lyon}\\ 0 & \text{Nantes} \end{cases} $$

et pour le statut :

$$ g_{\text{statut}}= \begin{cases} -0.4 & \text{Nouveau}\\ 0.2 & \text{Régulier}\\ 0.8 & \text{Fidèle}. \end{cases} $$

Enfin, la probabilité d'achat est

$$ \boxed{ p=\mathbb P(Y=1\mid X) =\frac{1}{1+\exp(-s)} } $$

et l'achat est généré selon

Y∼Bernoulli(p).
	​


In [ ]:
effet_revenu = np.clip((revenu - 2000) / 2500, 0, 1.5)
effet_visites = 1.4 / (1 + np.exp(-(visites - 5)))
effet_age = 1.5 * np.exp(-((age - 40) ** 2) / (2 * 12**2))
effet_distance = -0.09 * distance

effet_ville = pd.Series(ville).map({
    "Tours": 0.5, "Paris": 0.2, "Lyon": -0.1, "Nantes": 0.0
}).to_numpy()

effet_statut = pd.Series(statut).map({
    "Nouveau": -0.4, "Régulier": 0.2, "Fidèle": 0.8
}).to_numpy()

score = (-2.4 + 1.2*effet_revenu + effet_visites + effet_age
         + effet_distance + effet_ville + effet_statut)

prob_achat = 1 / (1 + np.exp(-score))
achat = np.random.binomial(1, prob_achat)

df["achat"] = achat
df.head()



In [ ]:
df_complet = df.copy()

df_complet["prob_achat"] = prob_achat
df_complet.head()

## 2. Explorer les données

Avant tout modèle, on regarde les dimensions, les types et les statistiques descriptives.

In [ ]:
print("Dimensions :", df.shape)
print(df.dtypes)
print()
df.describe().T

## 3. Histogrammes des variables

On compare les distributions des individus qui achètent et de ceux qui n'achètent pas.

L'objectif n'est pas de « prouver » une relation à partir d'un histogramme, mais de repérer des structures intéressantes.

In [ ]:
variables_numeriques = ["age", "revenu", "anciennete", "visites", "distance"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, variable in zip(axes.ravel(), variables_numeriques):
    ax.hist(df.loc[df["achat"] == 0, variable], bins=25,
            alpha=0.5, label="Pas d'achat")
    ax.hist(df.loc[df["achat"] == 1, variable], bins=25,
            alpha=0.5, label="Achat")
    ax.set_title(variable)
    ax.set_xlabel(variable)
    ax.set_ylabel("Effectif")
    ax.legend()

axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()

## 4. Variables catégorielles

On peut également examiner la fréquence des catégories et le taux d'achat associé.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["ville"].value_counts().plot(kind="bar", ax=axes[0])
axes[0].set_title("Répartition des villes")

df["statut"].value_counts().plot(kind="bar", ax=axes[1])
axes[1].set_title("Répartition des statuts")

plt.tight_layout()
plt.show()

print("Taux d'achat par ville :")
print(df.groupby("ville")["achat"].mean())

print("\nTaux d'achat par statut :")
print(df.groupby("statut")["achat"].mean())

## 5. Introduire des valeurs manquantes

Dans une vraie base, certaines valeurs sont absentes. Pour étudier leur traitement, nous supprimons volontairement environ 8 % des valeurs de certaines colonnes.

In [ ]:
rng = np.random.default_rng(123)

for colonne in ["age", "revenu", "anciennete", "ville"]:
    masque = rng.random(len(df)) < 0.08
    df.loc[masque, colonne] = np.nan

print(df.isna().sum())
print()
print("Taux de valeurs manquantes :")
print(df.isna().mean())

## 6. Que faire des données manquantes ?

Trois idées :

- supprimer les observations incomplètes ;
- remplacer les valeurs par une règle simple ;
- utiliser une méthode plus sophistiquée.

Pour les variables numériques, on peut utiliser la moyenne ou la médiane. Pour les catégories, on peut utiliser la modalité la plus fréquente.

`SimpleImputer` permet d'intégrer ces traitements dans `scikit-learn`.

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_demo = df[["age", "revenu", "anciennete"]]
X_demo_impute = imputer.fit_transform(X_demo)

print("Médianes apprises :", imputer.statistics_)
print(pd.DataFrame(X_demo_impute, columns=X_demo.columns).head())

### `fit` et `transform`

`fit` apprend la règle de transformation. Pour une imputation médiane, il apprend les médianes.

`transform` applique cette règle.

Le principe essentiel est :

```text
TRAIN : fit + transform
TEST  : transform seulement
```

On ne recalcule donc jamais la médiane sur le test.

## 7. Comparer moyenne et médiane

Une stratégie d'imputation peut être comparée comme n'importe quel choix de prétraitement.

La comparaison doit respecter la validation croisée : l'imputation est apprise séparément dans chaque fold.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

X_num = df[variables_numeriques]
y = df["achat"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modeles_imputation = {
    "Moyenne": Pipeline([
        ("imputation", SimpleImputer(strategy="mean")),
        ("modele", LogisticRegression(max_iter=2000))
    ]),
    "Médiane": Pipeline([
        ("imputation", SimpleImputer(strategy="median")),
        ("modele", LogisticRegression(max_iter=2000))
    ])
}

resultats = []
for nom, modele in modeles_imputation.items():
    scores = cross_val_score(modele, X_num, y, cv=cv, scoring="accuracy")
    resultats.append({
        "Stratégie": nom,
        "Accuracy moyenne": scores.mean(),
        "Écart-type": scores.std()
    })

pd.DataFrame(resultats)

## 8. `OneHotEncoder` : traiter les variables catégorielles

Un modèle ne doit généralement pas interpréter `Tours`, `Paris` et `Lyon` comme les nombres 0, 1 et 2 : cela créerait artificiellement un ordre.

Le one-hot encoding crée une variable binaire par catégorie.

`handle_unknown="ignore"` permet de gérer une catégorie rencontrée au moment de la prédiction mais absente de l'entraînement.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encodeur = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

ville_encodee = encodeur.fit_transform(df[["ville"]])

print(encodeur.get_feature_names_out())
print(ville_encodee[:5])

## 9. Mettre les variables numériques à l'échelle

`StandardScaler` réalise une centrage-réduction :

$
z = \frac{x-\mu}{\sigma}.
$

La moyenne et l'écart-type sont appris sur le train.

D'autres transformations existent :

- `MinMaxScaler` : ramène généralement les valeurs dans `[0,1]` ; Sensible aux outliers.
- `RobustScaler` : utilise la médiane et l'intervalle interquartile, donc résiste mieux aux valeurs extrêmes.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

revenu_sans_na = df[["revenu"]].dropna()

scalers = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler()
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (nom, scaler) in zip(axes, scalers.items()):
    x = scaler.fit_transform(revenu_sans_na)
    ax.hist(x.ravel(), bins=30)
    ax.set_title(nom)
    ax.set_xlabel("Valeur transformée")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# 1. Imputation
imputer = SimpleImputer(strategy="median")
revenu_impute = imputer.fit_transform(df[["revenu"]])

# 2. Les trois scalers
scalers = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler()
}

# 3. Visualisation
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (nom, scaler) in zip(axes, scalers.items()):
    x = scaler.fit_transform(revenu_impute)

    ax.hist(x.ravel(), bins=30)
    ax.set_title(nom)
    ax.set_xlabel("Valeur transformée")
    ax.set_ylabel("Effectif")

plt.tight_layout()
plt.show()

## 10. `ColumnTransformer`

Nous voulons appliquer des traitements différents selon les colonnes.

**Numériques :** imputation médiane → standardisation.

**Catégorielles :** imputation par modalité fréquente → one-hot encoding.

`ColumnTransformer` permet de réunir ces deux branches.

In [ ]:
from sklearn.compose import ColumnTransformer

variables_numeriques = ["age", "revenu", "anciennete", "visites", "distance"]
variables_categorielles = ["ville", "statut"]

preparation_numerique = Pipeline([
    ("imputation", SimpleImputer(strategy="median")),
    ("normalisation", StandardScaler())
])

preparation_categorielle = Pipeline([
    ("imputation", SimpleImputer(strategy="most_frequent")),
    ("encodage", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preparation = ColumnTransformer([
    ("numerique", preparation_numerique, variables_numeriques),
    ("categorielle", preparation_categorielle, variables_categorielles)
])

## 11. La `Pipeline` complète

On associe maintenant le prétraitement au modèle.

La pipeline apprend les transformations sur les données d'entraînement puis les réutilise sur les données nouvelles.

C'est aussi un moyen très efficace d'éviter le **data leakage**.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

X = df.drop(columns="achat")
y = df["achat"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

modele = Pipeline([
    ("preparation", preparation),
    ("classification", LogisticRegression(max_iter=2000))
])

modele.fit(X_train, y_train)

y_pred = modele.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

## 12. Pourquoi la pipeline protège contre le data leakage ?

Au moment de `fit(X_train, y_train)` :

1. les médianes sont calculées sur `X_train` ;
2. les paramètres du `StandardScaler` sont calculés sur `X_train` ;
3. les catégories du `OneHotEncoder` sont apprises sur `X_train` ;
4. le modèle est entraîné.

Au moment de `predict(X_test)`, aucune de ces quantités n'est recalculée sur le test.

Le test reste donc une donnée nouvelle pour le modèle.

## 13. Une dernière expérience : la relation est non linéaire

Nous avons construit les données avec des effets non linéaires.

On peut donc comparer la régression logistique à un modèle plus flexible.

Cette comparaison sert ici à illustrer une idée : le prétraitement ne suffit pas à résoudre tous les problèmes ; **la forme du modèle compte aussi**.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

modele_logistique = Pipeline([
    ("preparation", preparation),
    ("classification", LogisticRegression(max_iter=2000))
])

modele_foret = Pipeline([
    ("preparation", preparation),
    ("classification", RandomForestClassifier(
        n_estimators=300, random_state=42
    ))
])

modele_neurone = Pipeline([
    ("preparation", preparation),
    ("classification", MLPClassifier(
        hidden_layer_sizes=(10,),
        max_iter=2000,
        random_state=42
    ))
])

resultats_modeles = []

for nom, modele_test in {
    "Régression logistique": modele_logistique,
    "Random Forest": modele_foret,
    "Réseau de neurones": modele_neurone
}.items():
    modele_test.fit(X_train, y_train)
    resultats_modeles.append({
        "Modèle": nom,
        "Accuracy test": modele_test.score(X_test, y_test)
    })

pd.DataFrame(resultats_modeles)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

resultats_modeles = []

for nom, modele_test in {
    "Régression logistique": modele_logistique,
    "Random Forest": modele_foret,
    "Réseau de neurones": modele_neurone
}.items():
    
    modele_test.fit(X_train, y_train)

    # Prédictions
    y_pred_train = modele_test.predict(X_train)
    y_pred_test = modele_test.predict(X_test)

    resultats_modeles.append({
        "Modèle": nom,
        
        "Accuracy train": accuracy_score(y_train, y_pred_train),
        "Accuracy test": accuracy_score(y_test, y_pred_test),
        
        "Precision train": precision_score(y_train, y_pred_train),
        "Precision test": precision_score(y_test, y_pred_test),
        
        "Recall train": recall_score(y_train, y_pred_train),
        "Recall test": recall_score(y_test, y_pred_test),
        
        "F1 train": f1_score(y_train, y_pred_train),
        "F1 test": f1_score(y_test, y_pred_test)
    })

resultats_df = pd.DataFrame(resultats_modeles)
resultats_df

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

modeles = {
    "Régression logistique": modele_logistique,
    "Random Forest": modele_foret,
    "Réseau de neurones": modele_neurone
}

for nom, modele in modeles.items():

    ConfusionMatrixDisplay.from_estimator(
        modele,
        X_test,
        y_test
    )

    plt.title(f"Matrice de confusion — {nom}")
    plt.show()

In [ ]:
from sklearn.metrics import RocCurveDisplay

for nom, modele in modeles.items():
    RocCurveDisplay.from_estimator(
        modele,
        X_test,
        y_test,
        name=nom
    )

plt.plot([0, 1], [0, 1], "--")
plt.title("Courbes ROC")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
modeles = {
    "Régression logistique": modele_logistique,
    "Random Forest": modele_foret,
    "Réseau de neurones": modele_neurone
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (nom, modele) in zip(axes, modeles.items()):

    y_pred = modele.predict(X_test)

    sns.scatterplot(
        x=X_test["age"],
        y=X_test["revenu"],
        hue=(y_pred != y_test),
        ax=ax
    )

    ax.set_title(nom)
    ax.set_xlabel("Âge")
    ax.set_ylabel("Revenu")

plt.tight_layout()
plt.show()

In [ ]:
noms_variables = modele_foret.named_steps["preparation"].get_feature_names_out()

importances = modele_foret.named_steps["classification"].feature_importances_

importance_df = pd.DataFrame({
    "Variable": noms_variables,
    "Importance": importances
}).sort_values("Importance", ascending=False)

importance_df

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=importance_df,
    x="Importance",
    y="Variable"
)

plt.title("Importance des variables — Random Forest")
plt.tight_layout()
plt.show()

In [ ]:
modele_logistique.named_steps["classification"].coef_[0]

In [ ]:
noms_variables = modele_logistique.named_steps["preparation"].get_feature_names_out()

coefficients = modele_logistique.named_steps["classification"].coef_[0]

coef_df = pd.DataFrame({
    "Variable": noms_variables,
    "Coefficient": coefficients
}).sort_values("Coefficient")

# Bilan

Nous avons rencontré les principaux problèmes de préparation des données :

- **valeurs manquantes** → `SimpleImputer` ;
- **échelles différentes** → `StandardScaler` ;
- **valeurs extrêmes** → `RobustScaler` peut être utile ;
- **variables catégorielles** → `OneHotEncoder` ;
- **traitements différents selon les colonnes** → `ColumnTransformer` ;
- **enchaînement du prétraitement et du modèle** → `Pipeline` ;
- **séparation train/test** → les transformations sont apprises sur le train uniquement.

La structure générale à retenir :

```text
                    DONNÉES
                       │
             ┌─────────┴─────────┐
             │                   │
        numériques          catégorielles
             │                   │
       imputation           imputation
             │                   │
       normalisation        OneHotEncoder
             │                   │
             └─────────┬─────────┘
                       │
                ColumnTransformer
                       │
                    Pipeline
                       │
                     Modèle
                       │
                  Évaluation
```

### Transition

Notre jeu de données contient des relations non linéaires. Cela conduit naturellement à la question de la séance suivante :

> **Comment construire des modèles capables d'apprendre des relations complexes entre les variables et la cible ?**